# Notebook 03b: Element Classification

**Pipeline layer D.** Sits between Notebook 03 (stockiness measures) and Notebook 03c (network and combination analysis). Non-destructive: this notebook adds columns to a copy of the stockiness table and changes nothing in the original.

## Purpose

Notebook 03 computes four stockiness measures for every `(element, element_type, genre)` cell. The element vocabulary still contains two kinds of extraction noise that would pollute the rankings if left in: proper names that spaCy did not catch as `PERSON` entities (cartoon characters, franchise villains, Bollywood given names, named places and organizations), and vacuous generic words that the stopword list missed (`thing`, `way`, `part`, `situation`).

This notebook classifies every unique `(element, element_type)` pair into a ten-category taxonomy. Six categories are genuine narrative elements; four are noise to be filtered downstream. The output adds `element_taxonomy`, `is_narrative_element`, and supporting columns to the stockiness table.

## The genre-invariance assumption

A taxonomic category is treated as a property of the `(element, element_type)` pair, **not** of the `(element, element_type, genre)` cell. "Sheriff" is a `character_type` whether it appears in Western or Comedy. This is unlike stockiness, which is inherently genre-relative. The assumption holds for the large majority of elements. Genuinely polysemous tokens whose category depends on sense ("bar" as saloon versus gold bar) are flagged `is_ambiguous` and, where the senses are clearly split, `is_split_sense`, so downstream analyses can handle the flagged tail rather than trusting a single label.

Classifying per `(element, element_type, genre)` instead would cost roughly eleven times as much for redundant labels on the genre-invariant majority. That tradeoff is not worth it; the flagged-tail approach is.

## Method

A two-tier classifier with cross-checks.

**Tier 1** classifies every unique element from lightweight metadata (the element string, its spaCy `element_type`, the genres where it is most frequent, and its corpus counts). Fast and cheap.

**Tier 2** re-classifies only the flagged subset (low Tier 1 confidence, self-reported ambiguity, disagreement with a rule prior, or membership in a known-polysemous list) using four to six real example sentences pulled from the plot corpus. Slower per element, but applied only where context actually changes the answer.

The LLM is not the only signal. Two cross-checks run alongside it: the spaCy `element_type` as a rule prior (an entity tagged `GPE` is almost certainly a `place_name`), and inline name gazetteers. The cross-checks do not override the LLM; they route elements to Tier 2 and feed the validation step.

**Validation** is against a curated gold set of hand-labeled elements covering every category, the known leakage names this notebook exists to catch, the polysemous hard cases, chunks, verbs, and entities.

## Inputs

- `v4_rebuild/03_stockiness/stockiness_long.parquet` (the stockiness table, one row per element-type-genre cell)
- `v4_rebuild/02_elements/element_index.parquet` (element to film index, used for Tier 2 context retrieval)
- `v4_rebuild/01c_union/movies_union.parquet` (plot text, used for Tier 2 context retrieval)
- `ANTHROPIC_API_KEY` via Google Colab Secrets

## Outputs

- `v4_rebuild/03b_classification/element_classifications.parquet` (one row per unique element, with taxonomy and flags)
- `v4_rebuild/03b_classification/stockiness_long_classified.parquet` (the canonical enriched stockiness table that all downstream notebooks consume)
- `v4_rebuild/03b_classification/classification_report.txt`
- `v4_rebuild/03b_classification/cache_tier1/*.json`, `cache_tier2/*.json` (per-batch caches; re-running the notebook skips cached batches)

## Cost and runtime

Roughly 25 to 35 US dollars in Claude API charges depending on how large the Tier 2 set turns out to be, and about 30 to 45 minutes of wall time. Prompt caching on the taxonomy system prompt is enabled to hold the cost down. Set `DRY_RUN = True` in the config cell to test the full pipeline on two batches before committing to the full run.

In [2]:
# ----------------------------------------------------------------------
# Setup and configuration
# ----------------------------------------------------------------------
!pip install anthropic -q

import os, re, json, time, hashlib, random
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- Google Drive -----------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- Paths ------------------------------------------------------------
ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
            'DGK Lab - Scene A Faire Computational')
V4   = ROOT / 'Analysis/v4_rebuild'

PATH_STOCKINESS = V4 / '03_stockiness'  / 'stockiness_long.parquet'
PATH_ELEM_INDEX = V4 / '02_elements'    / 'element_index.parquet'
PATH_MOVIES     = V4 / '01c_union'      / 'movies_union.parquet'

OUT_DIR        = V4 / '03b_classification'
CACHE_T1       = OUT_DIR / 'cache_tier1'
CACHE_T2       = OUT_DIR / 'cache_tier2'
CACHE_CTX      = OUT_DIR / 'cache_contexts'
for d in (OUT_DIR, CACHE_T1, CACHE_T2, CACHE_CTX):
    d.mkdir(parents=True, exist_ok=True)

PATH_CLASSIFICATIONS = OUT_DIR / 'element_classifications.parquet'
PATH_ENRICHED        = OUT_DIR / 'stockiness_long_classified.parquet'
PATH_REPORT          = OUT_DIR / 'classification_report.txt'

# --- API key (never hardcode; Colab Secrets only) ---------------------
from google.colab import userdata
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    assert ANTHROPIC_API_KEY, 'empty'
except Exception:
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found in Colab Secrets. Open the key icon in "
        "the left sidebar, add a secret named ANTHROPIC_API_KEY, and enable "
        "notebook access. Do not paste the key into a cell."
    )

import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# --- Run configuration ------------------------------------------------
MODEL          = 'claude-sonnet-4-6'
DRY_RUN        = False    # True: process only the first 2 batches per tier
TIER1_BATCH    = 40       # elements per Tier 1 request
TIER2_BATCH    = 20       # elements per Tier 2 request (larger prompts)
MAX_WORKERS    = 4        # concurrent API calls; keep modest for rate limits
TIER2_CONTEXTS = 5        # example sentences per element in Tier 2
SEED           = 20260219 # matches the preregistration seed used elsewhere

random.seed(SEED)
np.random.seed(SEED)

print('Setup complete.')
print(f'  model        : {MODEL}')
print(f'  output dir   : {OUT_DIR}')
print(f'  DRY_RUN      : {DRY_RUN}')

Mounted at /content/drive
Setup complete.
  model        : claude-sonnet-4-6
  output dir   : /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/03b_classification
  DRY_RUN      : False


In [3]:
# ----------------------------------------------------------------------
# Load the stockiness table and introspect its schema.
# The v4 column names are detected at runtime rather than assumed, so this
# cell works regardless of the exact naming Notebook 03 used.
# ----------------------------------------------------------------------
stk = pd.read_parquet(PATH_STOCKINESS)
print(f'stockiness_long.parquet: {stk.shape[0]:,} rows x {stk.shape[1]} cols')
print('columns:', list(stk.columns))
print()
print(stk.head(3).to_string())
print()

def find_col(df, candidates, required=True, label=''):
    """Return the first column whose lowercased name matches a candidate."""
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    # substring fallback
    for cand in candidates:
        for c in df.columns:
            if cand.lower() in c.lower():
                return c
    if required:
        raise KeyError(
            f'Could not find a column for {label!r}. Tried {candidates}. '
            f'Available: {list(df.columns)}'
        )
    return None

COL_ELEMENT = find_col(stk, ['element'], label='element')
COL_TYPE    = find_col(stk, ['element_type', 'elem_type', 'type'], label='element_type')
COL_GENRE   = find_col(stk, ['genre'], label='genre')
COL_FILMS   = find_col(stk, ['n_films', 'film_count', 'films', 'f_e_g', 'n_film'],
                       required=False, label='per-genre film count')
COL_INST    = find_col(stk, ['count', 'n_instances', 'instances', 'c_e_g', 'instance_count'],
                       required=False, label='per-genre instance count')
COL_ENSEMBLE = find_col(stk, ['ensemble', 'ensemble_stockiness', 'ensemble_z'],
                        required=False, label='ensemble score')
COL_PRIMARY  = find_col(stk, ['primary_stockiness', 'primary', 'ps'],
                        required=False, label='primary stockiness')

print('Detected columns:')
for label, col in [('element', COL_ELEMENT), ('element_type', COL_TYPE),
                   ('genre', COL_GENRE), ('film count', COL_FILMS),
                   ('instance count', COL_INST), ('ensemble', COL_ENSEMBLE),
                   ('primary', COL_PRIMARY)]:
    print(f'  {label:<16}: {col}')

# Coerce the key columns to str on the stockiness table itself so that the
# join in Cell 15 cannot silently fail on a dtype mismatch.
stk[COL_ELEMENT] = stk[COL_ELEMENT].astype(str)
stk[COL_TYPE]    = stk[COL_TYPE].astype(str)

# Unique (element, element_type) pairs are the classification units.
pairs = (stk[[COL_ELEMENT, COL_TYPE]]
         .drop_duplicates()
         .reset_index(drop=True))
print()
print(f'Unique (element, element_type) pairs to classify: {len(pairs):,}')
print()
print('element_type value counts:')
print(pairs[COL_TYPE].value_counts().to_string())

assert len(pairs) > 0, 'No element pairs found; check the stockiness file.'

stockiness_long.parquet: 456,159 rows x 31 cols
columns: ['element', 'element_type', 'genre', 'instance_count', 'film_count', 'total_instance_count', 'total_film_count', 'n_genres_present', 'rel_freq_in_genre', 'distinctiveness', 'primary_stockiness', 'tf', 'idf', 'tfidf', 'pmi', 'pmi_positive', 'dunning_g2', 'dunning_g2_positive', 'posterior_stockiness', 'posterior_smoothed', 'p_prior_g', 'element_entropy', 'element_entropy_normalized', 'primary_stockiness_z', 'tfidf_z', 'dunning_g2_positive_z', 'posterior_smoothed_z', 'posterior_stockiness_z', 'ensemble', 'is_universal_strict', 'breadth_penalty']

                   element element_type      genre  instance_count  film_count  total_instance_count  total_film_count  n_genres_present  rel_freq_in_genre  distinctiveness  primary_stockiness        tf       idf     tfidf       pmi  pmi_positive  dunning_g2  dunning_g2_positive  posterior_stockiness  posterior_smoothed  p_prior_g  element_entropy  element_entropy_normalized  primary_stocki

In [4]:
# ----------------------------------------------------------------------
# Build the metadata table fed to the Tier 1 classifier.
# For each unique (element, element_type): total films, total instances,
# number of genres present, and the three genres where it is most frequent.
# All of this comes from stockiness_long; element_index is not needed here.
# Vectorized: two groupbys rather than a per-element Python loop.
# ----------------------------------------------------------------------
grp = stk.groupby([COL_ELEMENT, COL_TYPE], sort=False)

# scalar aggregates
meta = grp.agg(n_genres_present=(COL_GENRE, 'nunique')).reset_index()
if COL_FILMS:
    tf = grp[COL_FILMS].sum().reset_index(name='total_films')
    meta = meta.merge(tf, on=[COL_ELEMENT, COL_TYPE], how='left')
else:
    meta['total_films'] = np.nan
if COL_INST:
    ti = grp[COL_INST].sum().reset_index(name='total_instances')
    meta = meta.merge(ti, on=[COL_ELEMENT, COL_TYPE], how='left')
else:
    meta['total_instances'] = np.nan

# top three genres per element, ranked by per-genre film count when available
if COL_FILMS:
    ranked = stk.sort_values([COL_ELEMENT, COL_TYPE, COL_FILMS],
                             ascending=[True, True, False])
else:
    ranked = stk.sort_values([COL_ELEMENT, COL_TYPE])
top3 = (ranked.groupby([COL_ELEMENT, COL_TYPE], sort=False).head(3)
              .groupby([COL_ELEMENT, COL_TYPE], sort=False)[COL_GENRE]
              .apply(list).reset_index(name='top_genres'))
meta = meta.merge(top3, on=[COL_ELEMENT, COL_TYPE], how='left')

# normalize column names and dtypes
meta = meta.rename(columns={COL_ELEMENT: 'element', COL_TYPE: 'element_type'})
meta['element'] = meta['element'].astype(str)
meta['element_type'] = meta['element_type'].astype(str)
meta['n_genres_present'] = meta['n_genres_present'].astype(int)
meta['top_genres'] = meta['top_genres'].apply(
    lambda x: list(x) if isinstance(x, (list, np.ndarray)) else [])

print(f'metadata table: {meta.shape[0]:,} rows')
print()
print(meta.head(8).to_string())
print()
print('n_genres_present distribution:')
print(meta['n_genres_present'].describe().to_string())

metadata table: 40,877 rows

                   element    element_type  n_genres_present  total_films  total_instances                     top_genres
0  101st airborne division      entity_ORG                 8           25               25           [Drama, Action, War]
1            1st battalion      entity_ORG                 6           12               12          [War, History, Drama]
2        221b baker street      entity_FAC                 9           27               27       [Crime, Thriller, Drama]
3                      707  entity_PRODUCT                 7           10               13  [Action, Adventure, Thriller]
4                      747  entity_PRODUCT                 8           43               76  [Action, Thriller, Adventure]
5                      767  entity_PRODUCT                 6           10               15          [Drama, History, War]
6              7th cavalry      entity_ORG                 9           35               49      [Western, Drama, Hist

In [5]:
# ----------------------------------------------------------------------
# Gazetteers and rule priors. These are CROSS-CHECKS, not the primary
# classifier. A gazetteer hit or a strong element_type prior that disagrees
# with the LLM routes an element to Tier 2; it does not override the LLM.
#
# The lists below are intentionally inline and editable so a reviewer can
# see exactly what is being used as a name signal. They are not exhaustive.
# ----------------------------------------------------------------------

# Common given names (US census top names, abbreviated). Lowercased.
COMMON_FIRST_NAMES = set("""
james john robert michael william david richard joseph thomas charles
christopher daniel matthew anthony donald mark paul steven andrew kenneth
joshua kevin brian george edward ronald timothy jason jeffrey ryan jacob
gary nicholas eric jonathan stephen larry justin scott brandon benjamin
samuel gregory frank alexander raymond patrick jack dennis jerry tyler
aaron jose henry adam douglas nathan peter zachary kyle walter harold
jeremy ethan carl keith roger gerald christian terry sean arthur austin
noah lawrence jesse joe bryan billy jordan albert dylan bruce willie gabriel
alan juan logan wayne ralph roy eugene randy vincent russell louis philip
bobby johnny bradley mary patricia jennifer linda elizabeth barbara susan
jessica sarah karen nancy lisa margaret betty sandra ashley dorothy kimberly
emily donna michelle carol amanda melissa deborah stephanie rebecca laura
sharon cynthia kathleen amy angela shirley anna ruth brenda pamela nicole
katherine virginia catherine christine samantha debra janet rachel carolyn
emma maria heather diane julie joyce victoria kelly christina joan evelyn
lauren judith megan andrea cheryl hannah jacqueline martha gloria teresa
ann sara madison frances kathryn janice jean abigail alice julia judy sophia
grace denise amber doris marilyn danielle beverly isabella theresa diana
natalie brittany charlotte marie kayla alexis lori
""".split())

# Common Indian and Bollywood given names (the kind that leaked through).
INDIAN_NAMES = set("""
shekhar anand sanjana nandini govinda prem raj rahul priya amit anil sunil
vijay arjun karan rohan rohit raja kumar ravi suresh ramesh mohan deepak
sanjay ajay vikram aditya akash krishna shiva ganesh lakshmi sita radha
geeta meera kavita anjali pooja neha sneha divya isha tara maya rani devi
asha usha rekha hema kareena katrina deepika aishwarya madhuri sridevi
amitabh shahrukh salman aamir hrithik ranbir ranveer akshay ajay nawaz
vikrant babu chand mani gopal kishan laxman bharat
""".split())

# Curated franchise / fictional character names from the known leakage.
FRANCHISE_CHARACTERS = set("""
daffy tweety porky bugs sylvester elmer marvin taz speedy foghorn
pinhead leatherface jason freddy chucky michael ghostface candyman
coffinjoe coffin tallman pennywise jigsaw
moriarty drummond holmes watson poirot marple bond
voldemort dumbledore gandalf frodo sauron aragorn legolas dobby
rambo rocky terminator robocop predator alien xenomorph
godzilla mothra dracula frankenstein wolfman mummy
tarzan zorro batman superman spiderman joker
shrek donkey pinocchio aladdin simba mufasa scar
norman bates hannibal lecter
""".split())

NAME_GAZETTEER = COMMON_FIRST_NAMES | INDIAN_NAMES | FRANCHISE_CHARACTERS

# Known-polysemous tokens: words that are both a common noun and a name, or
# that otherwise carry more than one category-relevant sense. Always routed
# to Tier 2 regardless of Tier 1 confidence.
AMBIGUOUS_WORDS = set("""
rose will mark faith hope grace dawn june may april joy hunter summer
art bill jack patience charity prudence destiny justice chance harmony
star pearl ruby crystal violet iris daisy ivy lily heather holly
bar company state union order spring fall march page sage drew
amber jasmine angel cross trinity"""
.split())

def detect_type_family(element_type):
    """
    Map a raw element_type string to a coarse family, robust to whether the
    v4 pipeline stored entities as 'GPE' or 'entity_GPE' or 'ent_gpe'.
    Returns one of: noun, verb, chunk, ent_GPE, ent_LOC, ent_FAC, ent_ORG,
    ent_NORP, ent_EVENT, ent_WORK, ent_PRODUCT, ent_PERSON, other.
    """
    t = element_type.lower()
    if 'gpe' in t:          return 'ent_GPE'
    if 'work_of_art' in t or 'work' in t: return 'ent_WORK'
    if 'norp' in t:         return 'ent_NORP'
    if 'fac' in t:          return 'ent_FAC'
    if 'org' in t:          return 'ent_ORG'
    if 'product' in t:      return 'ent_PRODUCT'
    if 'event' in t:        return 'ent_EVENT'
    if 'loc' in t:          return 'ent_LOC'
    if 'person' in t:       return 'ent_PERSON'
    if 'chunk' in t or 'phrase' in t or 'noun_phrase' in t: return 'chunk'
    if t.startswith('verb') or t == 'v':  return 'verb'
    if t.startswith('noun') or t == 'n':  return 'noun'
    return 'other'

# Strong priors: (predicted category, strength). strength 'hard' means a
# disagreement with the LLM is a real red flag; 'soft' is a weak nudge.
TYPE_PRIOR = {
    'ent_GPE':     ('place_name',        'hard'),
    'ent_LOC':     ('place_name',        'soft'),
    'ent_FAC':     ('place_name',        'soft'),
    'ent_ORG':     ('proper_noun_other', 'soft'),
    'ent_WORK':    ('proper_noun_other', 'hard'),
    'ent_PRODUCT': ('object',            'soft'),
    'ent_EVENT':   ('event',             'soft'),
    'ent_NORP':    ('character_type',    'soft'),
    'ent_PERSON':  ('character_name',    'hard'),
    'verb':        ('event',             'soft'),
}

def rule_prior(element, element_type):
    """Return (predicted_category or None, strength or None)."""
    fam = detect_type_family(element_type)
    if fam in TYPE_PRIOR:
        return TYPE_PRIOR[fam]
    return (None, None)

def gazetteer_hit(element):
    """True if the element (single token) matches a name gazetteer."""
    e = element.strip().lower()
    if ' ' in e:
        return False
    return e in NAME_GAZETTEER

# quick smoke test of the helpers
for t in ['noun', 'VERB', 'entity_GPE', 'ent_WORK_OF_ART', 'chunk', 'NORP']:
    print(f'  {t:<18} -> family={detect_type_family(t):<12} prior={rule_prior("x", t)}')
print('  gazetteer hit "daffy":', gazetteer_hit('daffy'),
      '| "sheriff":', gazetteer_hit('sheriff'))

  noun               -> family=noun         prior=(None, None)
  VERB               -> family=verb         prior=('event', 'soft')
  entity_GPE         -> family=ent_GPE      prior=('place_name', 'hard')
  ent_WORK_OF_ART    -> family=ent_WORK     prior=('proper_noun_other', 'hard')
  chunk              -> family=chunk        prior=(None, None)
  NORP               -> family=ent_NORP     prior=('character_type', 'soft')
  gazetteer hit "daffy": True | "sheriff": False


In [6]:
# ----------------------------------------------------------------------
# Curated gold set. Hand-labeled elements with known correct categories,
# used to validate the classifier (Cell 14). Not circular: these labels are
# fixed before the run. For genuinely polysemous entries, `also_ok` lists
# alternative acceptable categories, and the validator also accepts the
# element being flagged ambiguous.
#
# `type_hint` is optional and matched loosely; most entries are
# unambiguous by string alone.
# ----------------------------------------------------------------------
GOLD = [
    # --- character_type --------------------------------------------------
    {'element': 'sheriff',    'cat': 'character_type'},
    {'element': 'detective',  'cat': 'character_type'},
    {'element': 'scientist',  'cat': 'character_type'},
    {'element': 'vampire',    'cat': 'character_type'},
    {'element': 'ghost',      'cat': 'character_type'},
    {'element': 'soldier',    'cat': 'character_type'},
    {'element': 'nurse',      'cat': 'character_type'},
    {'element': 'cowboy',     'cat': 'character_type'},
    {'element': 'outlaw',     'cat': 'character_type'},
    {'element': 'witch',      'cat': 'character_type'},
    {'element': 'robot',      'cat': 'character_type'},
    {'element': 'alien',      'cat': 'character_type'},
    {'element': 'zombie',     'cat': 'character_type'},
    {'element': 'priest',     'cat': 'character_type'},
    {'element': 'spy',        'cat': 'character_type'},
    {'element': 'pirate',     'cat': 'character_type'},
    {'element': 'gangster',   'cat': 'character_type'},
    {'element': 'henchman',   'cat': 'character_type'},
    {'element': 'gang',       'cat': 'character_type'},
    {'element': 'posse',      'cat': 'character_type'},
    # --- setting ---------------------------------------------------------
    {'element': 'saloon',     'cat': 'setting'},
    {'element': 'spaceship',  'cat': 'setting'},
    {'element': 'castle',     'cat': 'setting'},
    {'element': 'hospital',   'cat': 'setting'},
    {'element': 'prison',     'cat': 'setting'},
    {'element': 'courtroom',  'cat': 'setting'},
    {'element': 'forest',     'cat': 'setting'},
    {'element': 'mansion',    'cat': 'setting'},
    {'element': 'laboratory', 'cat': 'setting'},
    {'element': 'battlefield','cat': 'setting'},
    {'element': 'dungeon',    'cat': 'setting'},
    {'element': 'ranch',      'cat': 'setting'},
    {'element': 'swamp',      'cat': 'setting'},
    # --- object ----------------------------------------------------------
    {'element': 'sword',      'cat': 'object'},
    {'element': 'gun',        'cat': 'object'},
    {'element': 'potion',     'cat': 'object'},
    {'element': 'map',        'cat': 'object'},
    {'element': 'knife',      'cat': 'object'},
    {'element': 'treasure',   'cat': 'object'},
    {'element': 'crown',      'cat': 'object'},
    {'element': 'amulet',     'cat': 'object'},
    {'element': 'bomb',       'cat': 'object'},
    {'element': 'diary',      'cat': 'object'},
    # --- event -----------------------------------------------------------
    {'element': 'wedding',    'cat': 'event'},
    {'element': 'shootout',   'cat': 'event'},
    {'element': 'murder',     'cat': 'event'},
    {'element': 'robbery',    'cat': 'event'},
    {'element': 'battle',     'cat': 'event'},
    {'element': 'funeral',    'cat': 'event'},
    {'element': 'trial',      'cat': 'event'},
    {'element': 'kidnapping', 'cat': 'event'},
    {'element': 'duel',       'cat': 'event'},
    {'element': 'heist',      'cat': 'event'},
    {'element': 'ambush',     'cat': 'event'},
    {'element': 'investigate','cat': 'event', 'type_hint': 'verb'},
    {'element': 'betray',     'cat': 'event', 'type_hint': 'verb'},
    {'element': 'haunt',      'cat': 'event', 'type_hint': 'verb'},
    {'element': 'rescue',     'cat': 'event', 'type_hint': 'verb'},
    {'element': 'seduce',     'cat': 'event', 'type_hint': 'verb'},
    {'element': 'avenge',     'cat': 'event', 'type_hint': 'verb'},
    # --- theme -----------------------------------------------------------
    {'element': 'redemption', 'cat': 'theme'},
    {'element': 'revenge',    'cat': 'theme'},
    {'element': 'betrayal',   'cat': 'theme', 'also_ok': ['event']},
    {'element': 'sacrifice',  'cat': 'theme', 'also_ok': ['event']},
    {'element': 'loyalty',    'cat': 'theme'},
    {'element': 'forgiveness','cat': 'theme'},
    {'element': 'corruption', 'cat': 'theme'},
    {'element': 'survival',   'cat': 'theme'},
    {'element': 'freedom',    'cat': 'theme'},
    # --- relationship ----------------------------------------------------
    {'element': 'marriage',     'cat': 'relationship', 'also_ok': ['event']},
    {'element': 'friendship',   'cat': 'relationship'},
    {'element': 'rivalry',      'cat': 'relationship'},
    {'element': 'romance',      'cat': 'relationship', 'also_ok': ['theme']},
    {'element': 'partnership',  'cat': 'relationship'},
    {'element': 'mentorship',   'cat': 'relationship'},
    {'element': 'brotherhood',  'cat': 'relationship'},
    # --- chunks ----------------------------------------------------------
    {'element': 'haunted house',   'cat': 'setting',         'type_hint': 'chunk'},
    {'element': 'mad scientist',   'cat': 'character_type',  'type_hint': 'chunk'},
    {'element': 'femme fatale',    'cat': 'character_type',  'type_hint': 'chunk'},
    {'element': 'final girl',      'cat': 'character_type',  'type_hint': 'chunk'},
    {'element': 'time machine',    'cat': 'object',          'type_hint': 'chunk'},
    {'element': 'treasure map',    'cat': 'object',          'type_hint': 'chunk'},
    {'element': 'love triangle',   'cat': 'relationship',    'type_hint': 'chunk',
     'also_ok': ['theme']},
    {'element': 'evil twin',       'cat': 'character_type',  'type_hint': 'chunk'},
    {'element': 'secret identity', 'cat': 'theme',           'type_hint': 'chunk',
     'also_ok': ['object']},
    {'element': 'double cross',    'cat': 'event',           'type_hint': 'chunk'},
    # --- character_name (the leakage this notebook exists to catch) -------
    {'element': 'daffy',       'cat': 'character_name'},
    {'element': 'tweety',      'cat': 'character_name'},
    {'element': 'porky',       'cat': 'character_name'},
    {'element': 'pinhead',     'cat': 'character_name'},
    {'element': 'leatherface', 'cat': 'character_name'},
    {'element': 'moriarty',    'cat': 'character_name'},
    {'element': 'shekhar',     'cat': 'character_name'},
    {'element': 'anand',       'cat': 'character_name'},
    {'element': 'nandini',     'cat': 'character_name'},
    {'element': 'govinda',     'cat': 'character_name'},
    {'element': 'voldemort',   'cat': 'character_name'},
    {'element': 'rambo',       'cat': 'character_name'},
    {'element': 'tarzan',      'cat': 'character_name'},
    {'element': 'dracula',     'cat': 'character_name'},
    {'element': 'godzilla',    'cat': 'character_name'},
    # --- place_name ------------------------------------------------------
    {'element': 'london',       'cat': 'place_name'},
    {'element': 'paris',        'cat': 'place_name'},
    {'element': 'texas',        'cat': 'place_name'},
    {'element': 'chicago',      'cat': 'place_name'},
    {'element': 'tokyo',        'cat': 'place_name'},
    {'element': 'hogwarts',     'cat': 'place_name'},
    {'element': 'mordor',       'cat': 'place_name'},
    {'element': 'transylvania', 'cat': 'place_name'},
    {'element': 'manhattan',    'cat': 'place_name'},
    {'element': 'egypt',        'cat': 'place_name'},
    # --- proper_noun_other ----------------------------------------------
    {'element': 'fbi',       'cat': 'proper_noun_other'},
    {'element': 'cia',       'cat': 'proper_noun_other'},
    {'element': 'nasa',      'cat': 'proper_noun_other'},
    {'element': 'titanic',   'cat': 'proper_noun_other', 'also_ok': ['object']},
    {'element': 'christmas', 'cat': 'proper_noun_other', 'also_ok': ['event', 'setting']},
    # --- generic_or_empty ------------------------------------------------
    {'element': 'thing',     'cat': 'generic_or_empty'},
    {'element': 'way',       'cat': 'generic_or_empty'},
    {'element': 'lot',       'cat': 'generic_or_empty'},
    {'element': 'part',      'cat': 'generic_or_empty'},
    {'element': 'time',      'cat': 'generic_or_empty'},
    {'element': 'day',       'cat': 'generic_or_empty'},
    {'element': 'year',      'cat': 'generic_or_empty'},
    {'element': 'moment',    'cat': 'generic_or_empty'},
    {'element': 'fact',      'cat': 'generic_or_empty'},
    {'element': 'situation', 'cat': 'generic_or_empty'},
    {'element': 'man',       'cat': 'generic_or_empty'},
    {'element': 'woman',     'cat': 'generic_or_empty'},
    {'element': 'people',    'cat': 'generic_or_empty'},
    {'element': 'group',     'cat': 'generic_or_empty'},
    # --- polysemous hard cases (dominant sense + also_ok; amb flag also ok)
    {'element': 'rose',   'cat': 'object',           'also_ok': ['character_name'], 'poly': True},
    {'element': 'will',   'cat': 'generic_or_empty', 'also_ok': ['object', 'character_name'], 'poly': True},
    {'element': 'mark',   'cat': 'generic_or_empty', 'also_ok': ['object', 'character_name'], 'poly': True},
    {'element': 'faith',  'cat': 'theme',            'also_ok': ['character_name'], 'poly': True},
    {'element': 'hope',   'cat': 'theme',            'also_ok': ['character_name'], 'poly': True},
    {'element': 'hunter', 'cat': 'character_type',   'also_ok': ['character_name'], 'poly': True},
    {'element': 'dawn',   'cat': 'generic_or_empty', 'also_ok': ['setting', 'character_name'], 'poly': True},
    {'element': 'bar',    'cat': 'setting',          'also_ok': ['object'], 'poly': True},
    {'element': 'grace',  'cat': 'theme',            'also_ok': ['character_name'], 'poly': True},
    {'element': 'company','cat': 'character_type',   'also_ok': ['relationship', 'proper_noun_other'], 'poly': True},
]

gold_df = pd.DataFrame(GOLD)
print(f'Gold set: {len(gold_df)} hand-labeled elements')
print()
print('coverage by category:')
print(gold_df['cat'].value_counts().to_string())
print()
print(f'polysemous entries: {gold_df.get("poly", pd.Series(dtype=bool)).fillna(False).sum()}')

Gold set: 140 hand-labeled elements

coverage by category:
cat
character_type       26
event                18
generic_or_empty     17
setting              15
character_name       15
object               13
theme                13
place_name           10
relationship          8
proper_noun_other     5

polysemous entries: 10


/tmp/ipykernel_5415/1527542122.py:174: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  print(f'polysemous entries: {gold_df.get("poly", pd.Series(dtype=bool)).fillna(False).sum()}')


In [7]:
# ----------------------------------------------------------------------
# Taxonomy definition and prompt construction.
# The system prompt is held constant across all calls so it can be cached
# (Anthropic prompt caching), which is the main cost lever for this notebook.
# ----------------------------------------------------------------------

CATEGORIES = [
    'character_type', 'setting', 'object', 'event', 'theme', 'relationship',
    'character_name', 'place_name', 'proper_noun_other', 'generic_or_empty',
]
NARRATIVE_CATEGORIES = {
    'character_type', 'setting', 'object', 'event', 'theme', 'relationship',
}

SYSTEM_PROMPT = """You are classifying narrative elements extracted from movie plot summaries. Each element is a word or short phrase an NLP pipeline pulled from a plot. Assign each element to exactly one category describing its narrative function.

This supports a legal research project on the copyright doctrine of scenes a faire, which concerns whether narrative elements are generic conventions of a genre. The downstream analysis must separate genuine narrative elements from extraction noise.

GENUINE NARRATIVE ELEMENT CATEGORIES:

1. character_type - a kind or role of character, not a specific named individual. Examples: sheriff, scientist, vampire, mentor, soldier, detective, mad scientist, femme fatale. Collective character groups (gang, posse, crew) belong here. National, religious, or political groups functioning as character types in a story (pirates, Nazis) belong here.

2. setting - a kind of place or environment where action occurs. Examples: saloon, spaceship, hospital, forest, prison, castle, courtroom, haunted house. Generic, not a specific named place.

3. object - a physical prop, artifact, weapon, or item that figures in the plot. Examples: sword, gun, ring, map, potion, treasure, time machine.

4. event - a discrete action, incident, or beat in the plot. Examples: wedding, murder, shootout, escape, robbery, battle, investigation. Verbs describing plot actions (investigate, betray, rescue) belong here.

5. theme - an abstract concept, moral idea, or throughline. Examples: redemption, revenge, betrayal, sacrifice, justice, forbidden love.

6. relationship - a relational construct between characters. Examples: marriage, friendship, rivalry, mentorship, brotherhood, love triangle.

NON-NARRATIVE CATEGORIES (extraction noise to be filtered downstream):

7. character_name - a proper name of a specific character, real or fictional. Examples: Daffy, Sherlock, Voldemort, Shekhar, Rambo. If it names one particular character rather than a type, it belongs here even if you do not recognize the name.

8. place_name - a proper name of a specific real or fictional location. Examples: London, Texas, Hogwarts, Mordor, Transylvania.

9. proper_noun_other - any other proper noun: named organizations (FBI, NASA), titles of works, brands, products, named historical events (World War II), named vehicles (the Titanic).

10. generic_or_empty - a word that is not a proper noun but is too generic or vacuous to be a meaningful narrative element, or is an extraction artifact. Examples: thing, way, part, time, day, situation, man, woman, people, group, fact.

CLASSIFICATION RULES:
- Assign exactly one category: the dominant narrative function of the element.
- If the element has more than one common meaning falling in different categories (for example "rose" the flower versus the name Rose), still choose the single most likely dominant category, and set "amb" to true.
- A word being usable as a name does not by itself make it a character_name. "hunter" is usually a character_type; classify as character_name only when the specific-name reading clearly dominates.
- You are given the element_type tag from the NLP pipeline and the genres where the element is most frequent. Use them as evidence. Your judgment of narrative function determines the category.
- Be decisive. Set "conf" from 1 (low) to 5 (high).

OUTPUT FORMAT:
Return ONLY a JSON array. No prose, no markdown fences. One object per element, in the order given:
[{"e":"<element>","cat":"<category>","conf":<1-5>,"amb":<true|false>}]"""

SYSTEM_PROMPT_T2 = SYSTEM_PROMPT + """

ADDITIONAL CONTEXT FOR THIS PASS:
Each element below is accompanied by example sentences showing how it is actually used in the corpus. Use the examples to judge the dominant narrative function. If the example sentences show the element being used in two or more clearly distinct senses that fall in different categories, set "split" to true (in addition to "amb").

OUTPUT FORMAT FOR THIS PASS:
[{"e":"<element>","cat":"<category>","conf":<1-5>,"amb":<true|false>,"split":<true|false>}]"""


def fmt_element_line_t1(row):
    """One element line for a Tier 1 prompt."""
    genres = ', '.join(row['top_genres']) if row['top_genres'] else 'n/a'
    tf = row['total_films']
    tf_s = f'{int(tf)}' if pd.notna(tf) else 'n/a'
    return (f'- "{row["element"]}" | type={row["element_type"]} | '
            f'top_genres=[{genres}] | films={tf_s} | '
            f'genres_present={row["n_genres_present"]}')


def build_tier1_prompt(batch_df):
    lines = [fmt_element_line_t1(r) for _, r in batch_df.iterrows()]
    return ('Classify the following ' + str(len(lines)) +
            ' narrative elements.\n\n' + '\n'.join(lines))


def fmt_element_block_t2(row, contexts):
    """One element block for a Tier 2 prompt, with example sentences."""
    genres = ', '.join(row['top_genres']) if row['top_genres'] else 'n/a'
    head = (f'ELEMENT: "{row["element"]}" | type={row["element_type"]} | '
            f'top_genres=[{genres}]')
    if contexts:
        ctx = '\n'.join(f'    ex: {c}' for c in contexts)
    else:
        ctx = '    ex: (no example sentences found in corpus)'
    return head + '\n' + ctx


def build_tier2_prompt(batch_df, context_map):
    blocks = []
    for _, r in batch_df.iterrows():
        key = (r['element'], r['element_type'])
        blocks.append(fmt_element_block_t2(r, context_map.get(key, [])))
    return ('Classify the following ' + str(len(blocks)) +
            ' narrative elements, using the example sentences as evidence.'
            '\n\n' + '\n\n'.join(blocks))


print('Taxonomy:', len(CATEGORIES), 'categories')
print('  narrative    :', sorted(NARRATIVE_CATEGORIES))
print('  non-narrative:', sorted(set(CATEGORIES) - NARRATIVE_CATEGORIES))
print()
print('System prompt length:', len(SYSTEM_PROMPT), 'chars')

Taxonomy: 10 categories
  narrative    : ['character_type', 'event', 'object', 'relationship', 'setting', 'theme']
  non-narrative: ['character_name', 'generic_or_empty', 'place_name', 'proper_noun_other']

System prompt length: 3519 chars


In [8]:
# ----------------------------------------------------------------------
# API helper. One function handles both tiers. The system prompt is sent as
# a cacheable block. JSON is extracted defensively (fences stripped, array
# located). Retries with exponential backoff on transient errors.
# ----------------------------------------------------------------------

def _extract_json_array(text):
    """Pull the first JSON array out of a model response, tolerating fences."""
    t = text.strip()
    if t.startswith('```'):
        t = re.sub(r'^```(?:json)?\s*', '', t)
        t = re.sub(r'\s*```$', '', t)
    start = t.find('[')
    end   = t.rfind(']')
    if start == -1 or end == -1 or end < start:
        raise ValueError('no JSON array found in response')
    return json.loads(t[start:end + 1])


def call_classifier(system_prompt, user_prompt, max_tokens=4000, retries=5):
    """Call the model, return the parsed list of element dicts."""
    delay = 2.0
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=max_tokens,
                temperature=0,
                system=[{
                    'type': 'text',
                    'text': system_prompt,
                    'cache_control': {'type': 'ephemeral'},
                }],
                messages=[{'role': 'user', 'content': user_prompt}],
            )
            text = ''.join(
                block.text for block in resp.content
                if getattr(block, 'type', None) == 'text'
            )
            return _extract_json_array(text)
        except (anthropic.RateLimitError, anthropic.APIStatusError,
                anthropic.APIConnectionError, ValueError, json.JSONDecodeError) as e:
            last_err = e
            time.sleep(delay)
            delay = min(delay * 2, 60)
    raise RuntimeError(f'classifier call failed after {retries} attempts: {last_err}')


def _batch_indices(n, size):
    """Yield (batch_id, start, stop) over n items in fixed-size chunks."""
    for i, start in enumerate(range(0, n, size)):
        yield i, start, min(start + size, n)


def _parse_conf(value):
    """Coerce a model-reported confidence to an int in 1..5, tolerating
    ints, floats, and numeric strings; defaults to 1 on anything unusable."""
    try:
        c = int(round(float(value)))
    except (TypeError, ValueError):
        return 1
    return min(5, max(1, c))


print('API helper ready.')

API helper ready.


In [9]:
# ----------------------------------------------------------------------
# Tier 1: classify every unique element from metadata only.
# Deterministic batching (sorted elements, fixed batch size, cache keyed by
# batch id), so re-running the notebook skips completed batches.
# ----------------------------------------------------------------------

# Deterministic order: sort by (element, element_type).
meta_sorted = meta.sort_values(['element', 'element_type']).reset_index(drop=True)

n_elements = len(meta_sorted)
batches = list(_batch_indices(n_elements, TIER1_BATCH))
if DRY_RUN:
    batches = batches[:2]
    print(f'DRY_RUN: processing only {len(batches)} Tier 1 batches')

print(f'Tier 1: {n_elements:,} elements in {len(batches):,} batches '
      f'of {TIER1_BATCH}')


def _align_batch_response(batch_df, result, defaults, extra_bool_keys=()):
    """Align an LLM JSON-array response back to the prompt's batch rows.

    Why this is non-trivial: the output schema only carries "e", not
    "element_type". Roughly 14% of (element, element_type) pairs share an
    element string with a sibling row of a different type (the noun/verb
    overlap for words like "murder", and the entity/noun overlap for
    capitalized terms), and within a 40-row Tier 1 batch most of those
    sibling rows land together. A pure by-element dict collapses them,
    silently applying one classification to both rows.

    The fix: trust position first. With temperature=0 and the prompt
    instructing the model to return objects in input order, result[i]
    should correspond to batch_df.iloc[i]. We verify the model's "e" at
    position i matches the expected element; on mismatch we fall back to
    a by-element dict so a slightly reordered response still lands. The
    fallback still loses element_type for multi-POS collisions, but only
    when order has actually drifted.
    """
    by_elem = {str(r.get('e', '')).strip().lower(): r for r in result}
    aligned = []
    for i, (_, row) in enumerate(batch_df.iterrows()):
        expected = row['element'].strip().lower()
        chosen = None
        if i < len(result):
            cand = result[i]
            if str(cand.get('e', '')).strip().lower() == expected:
                chosen = cand
        if chosen is None:
            chosen = by_elem.get(expected)
        if chosen is None:
            chosen = {**defaults, '_unmatched': True}
        rec = {
            'element': row['element'],
            'element_type': row['element_type'],
            'cat':  chosen.get('cat', defaults['cat']),
            'conf': _parse_conf(chosen.get('conf', defaults['conf'])),
            'amb':  bool(chosen.get('amb', defaults['amb'])),
            'unmatched': bool(chosen.get('_unmatched', False)),
        }
        for k in extra_bool_keys:
            rec[k] = bool(chosen.get(k, defaults.get(k, False)))
        aligned.append(rec)
    return aligned


def _run_tier1_batch(batch_id, start, stop):
    cache_file = CACHE_T1 / f'batch_{batch_id:05d}.json'
    if cache_file.exists():
        return batch_id, json.loads(cache_file.read_text())
    batch_df = meta_sorted.iloc[start:stop]
    prompt = build_tier1_prompt(batch_df)
    result = call_classifier(SYSTEM_PROMPT, prompt, max_tokens=4000)
    aligned = _align_batch_response(
        batch_df, result,
        defaults={'cat': 'generic_or_empty', 'conf': 1, 'amb': True},
    )
    cache_file.write_text(json.dumps(aligned))
    return batch_id, aligned

tier1_records = {}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(_run_tier1_batch, bid, s, e): bid
               for bid, s, e in batches}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='tier1'):
        bid, aligned = fut.result()
        for rec in aligned:
            tier1_records[(rec['element'], rec['element_type'])] = rec

t1 = pd.DataFrame(tier1_records.values())
print()
print(f'Tier 1 complete: {len(t1):,} elements classified')
print()
print('Tier 1 category distribution:')
print(t1['cat'].value_counts().to_string())
bad_cats = set(t1['cat']) - set(CATEGORIES)
if bad_cats:
    print()
    print(f'WARNING: unexpected category labels from the model: {bad_cats}')
print()
print(f'unmatched (model dropped from batch): {int(t1["unmatched"].sum())}')
print(f'low confidence (conf <= 2)         : {int((t1["conf"] <= 2).sum())}')
print(f'self-reported ambiguous            : {int(t1["amb"].sum())}')

Tier 1: 40,877 elements in 1,022 batches of 40


tier1:   0%|          | 0/1022 [00:00<?, ?it/s]


Tier 1 complete: 40,877 elements classified

Tier 1 category distribution:
cat
character_type       7594
event                7023
generic_or_empty     6920
object               4561
setting              3310
character_name       3197
theme                3005
place_name           2917
proper_noun_other    1568
relationship          782

unmatched (model dropped from batch): 0
low confidence (conf <= 2)         : 1550
self-reported ambiguous            : 10029


In [10]:
# ----------------------------------------------------------------------
# Decide which elements go to Tier 2 for context-grounded re-classification.
# An element is flagged if ANY of the following holds:
#   - Tier 1 confidence is low (conf < 4)
#   - Tier 1 self-reported ambiguity
#   - the model dropped it from its batch (unmatched)
#   - it is in the known-polysemous AMBIGUOUS_WORDS list
#   - a HARD rule prior disagrees with the Tier 1 category
#   - a name gazetteer hit disagrees with the Tier 1 category
# ----------------------------------------------------------------------
def tier2_reason(row):
    reasons = []
    if row['unmatched']:
        reasons.append('unmatched')
    if row['conf'] < 4:
        reasons.append('low_conf')
    if row['amb']:
        reasons.append('self_amb')
    if row['element'].strip().lower() in AMBIGUOUS_WORDS:
        reasons.append('known_poly')
    prior_cat, strength = rule_prior(row['element'], row['element_type'])
    if prior_cat and strength == 'hard' and prior_cat != row['cat']:
        reasons.append(f'prior_conflict({prior_cat})')
    if gazetteer_hit(row['element']) and row['cat'] != 'character_name':
        reasons.append('gazetteer_conflict')
    return reasons

t1['tier2_reasons'] = t1.apply(tier2_reason, axis=1)
t1['needs_tier2']   = t1['tier2_reasons'].map(len) > 0

n_t2 = int(t1['needs_tier2'].sum())
print(f'Elements routed to Tier 2: {n_t2:,} of {len(t1):,} '
      f'({100 * n_t2 / len(t1):.1f}%)')
print()

reason_counter = Counter()
for reasons in t1.loc[t1['needs_tier2'], 'tier2_reasons']:
    for r in reasons:
        reason_counter[r.split('(')[0]] += 1
print('Tier 2 routing reasons (an element can have several):')
for reason, count in reason_counter.most_common():
    print(f'  {reason:<22}: {count:,}')

Elements routed to Tier 2: 16,339 of 40,877 (40.0%)

Tier 2 routing reasons (an element can have several):
  low_conf              : 14,605
  self_amb              : 10,029
  prior_conflict        : 295
  known_poly            : 90
  gazetteer_conflict    : 51


In [11]:
# ----------------------------------------------------------------------
# For each Tier 2 element, pull a few example sentences from the plot
# corpus. element_index gives the films an element appears in; movies_union
# gives the plot text. Matching is stem-aware and approximate: the goal is
# illustrative usage, not perfect recall.
# ----------------------------------------------------------------------
t2_keys = set(zip(t1.loc[t1['needs_tier2'], 'element'],
                  t1.loc[t1['needs_tier2'], 'element_type']))

ctx_cache_file = CACHE_CTX / 'contexts.json'
if ctx_cache_file.exists():
    raw = json.loads(ctx_cache_file.read_text())
    context_map = {tuple(k.split('\t')): v for k, v in raw.items()}
    print(f'Loaded {len(context_map):,} cached context sets')
else:
    print('Loading element_index and movies_union for context retrieval...')
    eidx = pd.read_parquet(PATH_ELEM_INDEX)
    ei_elem = find_col(eidx, ['element'], label='element')
    ei_type = find_col(eidx, ['element_type', 'elem_type', 'type'], label='element_type')
    ei_wiki = find_col(eidx, ['union_id', 'wiki_id', 'film_id', 'movie_id'], label='film id')

    movies = pd.read_parquet(PATH_MOVIES)
    mv_wiki = find_col(movies, ['union_id', 'wiki_id', 'film_id', 'movie_id'], label='film id')
    mv_plot = find_col(movies, ['plot', 'plot_text', 'summary', 'text'], label='plot')
    plot_by_id = dict(zip(movies[mv_wiki].astype(str), movies[mv_plot].astype(str)))

    # films per (element, element_type), restricted to the Tier 2 set
    eidx[ei_elem] = eidx[ei_elem].astype(str)
    eidx[ei_type] = eidx[ei_type].astype(str)
    eidx['_key'] = list(zip(eidx[ei_elem], eidx[ei_type]))
    eidx_t2 = eidx[eidx['_key'].isin(t2_keys)]
    films_by_key = (eidx_t2.groupby('_key')[ei_wiki]
                    .apply(lambda s: list(s.astype(str))[:40]).to_dict())

    # sanity check: the film ids in element_index must resolve against the
    # plot table. If almost nothing resolves, the two files key on different
    # id spaces and Tier 2 would silently get no context.
    _sample_ids = [wid for ids in list(films_by_key.values())[:200] for wid in ids]
    if _sample_ids:
        _hit = sum(1 for wid in _sample_ids if wid in plot_by_id) / len(_sample_ids)
        print(f'  film-id resolution check: {_hit:.0%} of sampled element_index '
              f'ids resolve to a plot')
        if _hit < 0.5:
            print('  WARNING: element_index and movies_union appear to use '
                  'different id columns. Tier 2 context will be sparse. '
                  f'(element_index id="{ei_wiki}", movies id="{mv_wiki}")')

    SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

    def find_sentences(element, plot, max_hits):
        """Return up to max_hits ~30-word windows containing the element."""
        toks = [w for w in re.split(r'\s+', element.strip().lower()) if w]
        if not toks:
            return []
        hits = []
        for sent in SENT_SPLIT.split(plot):
            low = sent.lower()
            if len(toks) == 1:
                # word-boundary, stem-tolerant (allow trailing s/es/ed/ing)
                pat = r'\b' + re.escape(toks[0]) + r'(?:s|es|ed|ing|d)?\b'
                ok = re.search(pat, low) is not None
            else:
                ok = all(re.search(r'\b' + re.escape(t), low) for t in toks)
            if ok:
                words = sent.split()
                if len(words) > 34:
                    # center the window on the first matched token
                    idx = next((i for i, w in enumerate(words)
                                if toks[0] in w.lower()), 0)
                    lo = max(0, idx - 16)
                    words = words[lo:lo + 32]
                hits.append(' '.join(words).strip())
                if len(hits) >= max_hits:
                    break
        return hits

    context_map = {}
    for key in tqdm(t2_keys, desc='contexts'):
        element, etype = key
        wiki_ids = films_by_key.get(key, [])
        collected = []
        for wid in wiki_ids:
            plot = plot_by_id.get(str(wid))
            if not plot:
                continue
            for s in find_sentences(element, plot, TIER2_CONTEXTS - len(collected)):
                collected.append(s)
            if len(collected) >= TIER2_CONTEXTS:
                break
        context_map[key] = collected

    ctx_cache_file.write_text(json.dumps(
        {f'{k[0]}\t{k[1]}': v for k, v in context_map.items()}))
    del eidx, eidx_t2, movies  # free memory

n_with_ctx = sum(1 for v in context_map.values() if v)
print(f'Context retrieval complete: {n_with_ctx:,} of {len(context_map):,} '
      f'Tier 2 elements have at least one example sentence')

Loading element_index and movies_union for context retrieval...
  film-id resolution check: 100% of sampled element_index ids resolve to a plot


contexts:   0%|          | 0/16339 [00:00<?, ?it/s]

Context retrieval complete: 16,264 of 16,339 Tier 2 elements have at least one example sentence


In [12]:
# ----------------------------------------------------------------------
# Tier 2: re-classify the flagged elements with example sentences.
# Results overwrite the Tier 1 record for those elements.
# ----------------------------------------------------------------------
t2_df = (meta_sorted[meta_sorted.apply(
            lambda r: (r['element'], r['element_type']) in t2_keys, axis=1)]
         .sort_values(['element', 'element_type']).reset_index(drop=True))

n_t2_elements = len(t2_df)
t2_batches = list(_batch_indices(n_t2_elements, TIER2_BATCH))
if DRY_RUN:
    t2_batches = t2_batches[:2]
    print(f'DRY_RUN: processing only {len(t2_batches)} Tier 2 batches')

print(f'Tier 2: {n_t2_elements:,} elements in {len(t2_batches):,} batches '
      f'of {TIER2_BATCH}')


def _run_tier2_batch(batch_id, start, stop):
    cache_file = CACHE_T2 / f'batch_{batch_id:05d}.json'
    if cache_file.exists():
        return batch_id, json.loads(cache_file.read_text())
    batch_df = t2_df.iloc[start:stop]
    prompt = build_tier2_prompt(batch_df, context_map)
    result = call_classifier(SYSTEM_PROMPT_T2, prompt, max_tokens=4000)
    aligned = _align_batch_response(
        batch_df, result,
        defaults={'cat': 'generic_or_empty', 'conf': 1, 'amb': True,
                  'split': False},
        extra_bool_keys=('split',),
    )
    cache_file.write_text(json.dumps(aligned))
    return batch_id, aligned

tier2_records = {}
if n_t2_elements > 0:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(_run_tier2_batch, bid, s, e): bid
                   for bid, s, e in t2_batches}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='tier2'):
            bid, aligned = fut.result()
            for rec in aligned:
                tier2_records[(rec['element'], rec['element_type'])] = rec

t2 = pd.DataFrame(tier2_records.values()) if tier2_records else pd.DataFrame(
    columns=['element', 'element_type', 'cat', 'conf', 'amb', 'split', 'unmatched'])
print()
print(f'Tier 2 complete: {len(t2):,} elements re-classified')
if len(t2):
    print()
    print('Tier 2 category distribution:')
    print(t2['cat'].value_counts().to_string())
    print()
    print(f'split-sense elements: {int(t2["split"].sum())}')

Tier 2: 16,339 elements in 817 batches of 20


tier2:   0%|          | 0/817 [00:00<?, ?it/s]


Tier 2 complete: 16,339 elements re-classified

Tier 2 category distribution:
cat
event                4001
character_name       3334
object               2228
generic_or_empty     1598
character_type       1526
theme                1339
setting               846
proper_noun_other     685
place_name            600
relationship          182

split-sense elements: 2845


In [13]:
# ----------------------------------------------------------------------
# Merge Tier 1 and Tier 2 into the final classification table.
# Tier 2 results take precedence for the elements that went through it.
# ----------------------------------------------------------------------
t2_lookup = {(r['element'], r['element_type']): r for _, r in t2.iterrows()}

final_rows = []
for _, r in t1.iterrows():
    key = (r['element'], r['element_type'])
    if key in t2_lookup:
        t2r = t2_lookup[key]
        cat, conf, amb = t2r['cat'], t2r['conf'], t2r['amb']
        split = bool(t2r.get('split', False))
        tier = 2
    else:
        cat, conf, amb = r['cat'], r['conf'], r['amb']
        split = False
        tier = 1

    prior_cat, strength = rule_prior(r['element'], r['element_type'])
    prior_agree = (prior_cat == cat) if prior_cat else None
    prior_hard_conflict = bool(prior_cat and strength == 'hard'
                               and prior_cat != cat)
    gaz_conflict = bool(gazetteer_hit(r['element']) and cat != 'character_name')

    final_rows.append({
        'element': r['element'],
        'element_type': r['element_type'],
        'element_taxonomy': cat,
        'is_narrative_element': cat in NARRATIVE_CATEGORIES,
        'classification_confidence': conf,
        'is_ambiguous': bool(amb),
        'is_split_sense': split,
        'classification_tier': tier,
        'rule_prior': prior_cat,
        'rule_prior_agrees': prior_agree,
        'needs_review': prior_hard_conflict or gaz_conflict,
    })

classifications = pd.DataFrame(final_rows)

# guardrail: every category the model produced should be in the taxonomy
bad = set(classifications['element_taxonomy']) - set(CATEGORIES)
if bad:
    print(f'WARNING: non-taxonomy labels present, coercing to generic_or_empty: {bad}')
    classifications.loc[
        classifications['element_taxonomy'].isin(bad), 'element_taxonomy'
    ] = 'generic_or_empty'
    classifications['is_narrative_element'] = (
        classifications['element_taxonomy'].isin(NARRATIVE_CATEGORIES))

classifications.to_parquet(PATH_CLASSIFICATIONS, index=False)
print(f'Saved element_classifications.parquet: {len(classifications):,} rows')
print()
print('Final taxonomy distribution:')
dist = classifications['element_taxonomy'].value_counts()
for cat in CATEGORIES:
    n = int(dist.get(cat, 0))
    flag = 'narrative' if cat in NARRATIVE_CATEGORIES else 'FILTERED'
    print(f'  {cat:<20} {n:>7,}  [{flag}]')
print()
print(f'is_narrative_element = True : '
      f'{int(classifications["is_narrative_element"].sum()):,}')
print(f'is_narrative_element = False: '
      f'{int((~classifications["is_narrative_element"]).sum()):,}')
print(f'ambiguous flagged           : {int(classifications["is_ambiguous"].sum()):,}')
print(f'split-sense flagged         : {int(classifications["is_split_sense"].sum()):,}')
print(f'needs_review (prior/gaz)    : {int(classifications["needs_review"].sum()):,}')
print(f'resolved at Tier 2          : '
      f'{int((classifications["classification_tier"] == 2).sum()):,}')

Saved element_classifications.parquet: 40,877 rows

Final taxonomy distribution:
  character_type         7,572  [narrative]
  setting                3,375  [narrative]
  object                 5,166  [narrative]
  event                  8,098  [narrative]
  theme                  3,056  [narrative]
  relationship             799  [narrative]
  character_name         4,364  [FILTERED]
  place_name             2,745  [FILTERED]
  proper_noun_other      1,433  [FILTERED]
  generic_or_empty       4,269  [FILTERED]

is_narrative_element = True : 28,066
is_narrative_element = False: 12,811
ambiguous flagged           : 5,844
split-sense flagged         : 2,845
needs_review (prior/gaz)    : 545
resolved at Tier 2          : 16,339


In [14]:
# ----------------------------------------------------------------------
# Validation against the curated gold set. Not circular: the gold labels
# were fixed before the run. For polysemous entries, a prediction counts as
# correct if it matches the primary category, matches an also_ok category,
# or the element was flagged ambiguous.
# ----------------------------------------------------------------------
cls_by_elem = defaultdict(list)
for _, r in classifications.iterrows():
    cls_by_elem[r['element'].strip().lower()].append(r)

def match_gold(entry):
    """Return (found, predicted_row or None)."""
    cands = cls_by_elem.get(entry['element'].strip().lower(), [])
    if not cands:
        return False, None
    hint = entry.get('type_hint')
    if hint:
        for c in cands:
            if hint.lower() in c['element_type'].lower():
                return True, c
    return True, cands[0]

results = []
for entry in GOLD:
    found, pred = match_gold(entry)
    if not found:
        results.append({**entry, 'status': 'not_in_corpus', 'predicted': None})
        continue
    ok_cats = {entry['cat']} | set(entry.get('also_ok', []))
    correct = (pred['element_taxonomy'] in ok_cats) or \
              (entry.get('poly', False) and pred['is_ambiguous'])
    results.append({
        'element': entry['element'],
        'expected': entry['cat'],
        'predicted': pred['element_taxonomy'],
        'confidence': pred['classification_confidence'],
        'ambiguous': pred['is_ambiguous'],
        'tier': pred['classification_tier'],
        'status': 'correct' if correct else 'WRONG',
    })

res_df = pd.DataFrame(results)
in_corpus = res_df[res_df['status'] != 'not_in_corpus']
n_correct = int((in_corpus['status'] == 'correct').sum())
n_total   = len(in_corpus)
acc = n_correct / n_total if n_total else float('nan')

print(f'Gold set validation: {n_correct}/{n_total} correct  '
      f'(accuracy {acc:.1%})')
not_found = res_df[res_df['status'] == 'not_in_corpus']
if len(not_found):
    print(f'  ({len(not_found)} gold elements not present in this corpus: '
          f'{", ".join(not_found["element"].tolist())})')
print()

# the central check: known leakage names must be filtered out
leakage = ['daffy', 'tweety', 'porky', 'pinhead', 'leatherface', 'moriarty',
           'shekhar', 'anand', 'nandini', 'govinda', 'voldemort', 'rambo']
leak_rows = classifications[
    classifications['element'].str.strip().str.lower().isin(leakage)]
leak_caught = leak_rows[~leak_rows['is_narrative_element']]
print(f'Known leakage names present in corpus: {len(leak_rows)}')
print(f'  filtered (is_narrative_element=False): {len(leak_caught)}')
if len(leak_rows) and len(leak_caught) < len(leak_rows):
    missed = leak_rows[leak_rows['is_narrative_element']]
    print(f'  WARNING: NOT filtered: '
          f'{", ".join(missed["element"].tolist())}')
print()

wrong = in_corpus[in_corpus['status'] == 'WRONG']
if len(wrong):
    print(f'Misclassifications ({len(wrong)}):')
    print(wrong[['element', 'expected', 'predicted', 'confidence',
                 'ambiguous', 'tier']].to_string(index=False))
print()

if np.isnan(acc):
    print('=' * 60)
    print('NOTE: gold-set accuracy is uncomputable — none of the curated')
    print('gold elements are present in the classified subset. This is')
    print('expected in DRY_RUN mode, where only the first 80 alphabetically')
    print('sorted elements are processed and the gold set sits later in')
    print('the vocabulary. The threshold check is meaningful only on a')
    print('full run.')
    print('=' * 60)
elif acc < 0.85:
    print('=' * 60)
    print('WARNING: gold-set accuracy is below 0.85. Inspect the')
    print('misclassifications above before trusting downstream output.')
    print('=' * 60)
else:
    print('Gold-set accuracy is acceptable.')

Gold set validation: 121/128 correct  (accuracy 94.5%)
  (12 gold elements not present in this corpus: mentorship, final girl, evil twin, voldemort, rambo, tarzan, mordor, thing, lot, moment, fact, people)

Known leakage names present in corpus: 18
  filtered (is_narrative_element=False): 18

Misclassifications (7):
    element         expected         predicted  confidence ambiguous  tier
  spaceship          setting            object         5.0     False   1.0
     castle          setting    character_name         3.0      True   2.0
      crown           object    character_name         4.0      True   2.0
 friendship     relationship             theme         5.0     False   1.0
brotherhood     relationship proper_noun_other         4.0      True   2.0
       time generic_or_empty            object         3.0      True   2.0
        man generic_or_empty    character_type         3.0      True   2.0

Gold-set accuracy is acceptable.


In [15]:
# ----------------------------------------------------------------------
# Join the classifications onto stockiness_long to produce the canonical
# enriched table. This is a left join on (element, element_type); the row
# count must not change and no classification column may be null afterward.
# ----------------------------------------------------------------------
join_cols = ['element', 'element_type', 'element_taxonomy',
             'is_narrative_element', 'classification_confidence',
             'is_ambiguous', 'is_split_sense', 'classification_tier',
             'needs_review']

cls_join = classifications[join_cols].rename(
    columns={'element': COL_ELEMENT, 'element_type': COL_TYPE})

n_before = len(stk)
enriched = stk.merge(cls_join, on=[COL_ELEMENT, COL_TYPE], how='left')

assert len(enriched) == n_before, (
    f'row count changed in join: {n_before:,} -> {len(enriched):,}')

n_unclassified = int(enriched['element_taxonomy'].isna().sum())
if n_unclassified:
    print(f'WARNING: {n_unclassified:,} stockiness rows did not receive a '
          f'classification. Filling with generic_or_empty / not-narrative.')
    enriched['element_taxonomy'] = enriched['element_taxonomy'].fillna('generic_or_empty')
    enriched['is_narrative_element'] = enriched['is_narrative_element'].fillna(False)
    enriched['classification_confidence'] = enriched['classification_confidence'].fillna(1)
    enriched['is_ambiguous'] = enriched['is_ambiguous'].fillna(True)
    enriched['is_split_sense'] = enriched['is_split_sense'].fillna(False)
    enriched['classification_tier'] = enriched['classification_tier'].fillna(0)
    enriched['needs_review'] = enriched['needs_review'].fillna(True)
else:
    print('All stockiness rows received a classification.')

enriched.to_parquet(PATH_ENRICHED, index=False)
print()
print(f'Saved stockiness_long_classified.parquet: {len(enriched):,} rows '
      f'x {enriched.shape[1]} cols')
print()
print('Share of stockiness rows that are genuine narrative elements:')
print(f'  {enriched["is_narrative_element"].mean():.1%}')

All stockiness rows received a classification.

Saved stockiness_long_classified.parquet: 456,159 rows x 38 cols

Share of stockiness rows that are genuine narrative elements:
  70.1%


In [16]:
# ----------------------------------------------------------------------
# Diagnostics. The key visual is the per-genre top-15 before and after the
# is_narrative_element filter: character names and junk should drop out.
# Also writes classification_report.txt.
# ----------------------------------------------------------------------
report_lines = []
def out(line=''):
    print(line)
    report_lines.append(line)

out('=' * 70)
out('NOTEBOOK 03b: ELEMENT CLASSIFICATION REPORT')
out('=' * 70)
out(f'unique elements classified : {len(classifications):,}')
out(f'stockiness rows enriched   : {len(enriched):,}')
out(f'gold-set accuracy          : {acc:.1%} ({n_correct}/{n_total})')
out('')
out('Taxonomy distribution (unique elements):')
for cat in CATEGORIES:
    n = int((classifications["element_taxonomy"] == cat).sum())
    flag = 'narrative' if cat in NARRATIVE_CATEGORIES else 'FILTERED'
    out(f'  {cat:<20} {n:>7,}  [{flag}]')
out('')
out(f'flagged ambiguous   : {int(classifications["is_ambiguous"].sum()):,}')
out(f'flagged split-sense : {int(classifications["is_split_sense"].sum()):,}')
out(f'flagged needs_review: {int(classifications["needs_review"].sum()):,}')
out(f'resolved at Tier 2  : '
    f'{int((classifications["classification_tier"] == 2).sum()):,}')
out('')

# rank column for the before/after view
rank_metric = COL_ENSEMBLE or COL_PRIMARY
if rank_metric:
    out('=' * 70)
    out(f'PER-GENRE TOP 15 BEFORE vs AFTER FILTER (ranked by {rank_metric})')
    out('=' * 70)
    genres = sorted(enriched[COL_GENRE].dropna().unique())
    for genre in genres:
        g = enriched[enriched[COL_GENRE] == genre]
        before = (g.sort_values(rank_metric, ascending=False)
                    .head(15)[COL_ELEMENT].tolist())
        after = (g[g['is_narrative_element']]
                   .sort_values(rank_metric, ascending=False)
                   .head(15)[COL_ELEMENT].tolist())
        dropped = [e for e in before if e not in after]
        out('')
        out(f'--- {genre} ---')
        out(f'  before: {", ".join(map(str, before))}')
        out(f'  after : {", ".join(map(str, after))}')
        if dropped:
            out(f'  dropped from top 15: {", ".join(map(str, dropped))}')
else:
    out('(no ensemble or primary score column detected; '
        'skipping before/after view)')

out('')
out('=' * 70)
out('ELEMENTS FLAGGED needs_review (rule-prior or gazetteer conflict)')
out('=' * 70)
review = classifications[classifications['needs_review']].head(40)
for _, r in review.iterrows():
    out(f'  "{r["element"]}" ({r["element_type"]}) -> '
        f'{r["element_taxonomy"]} | prior={r["rule_prior"]} | '
        f'conf={r["classification_confidence"]}')
if int(classifications['needs_review'].sum()) > 40:
    out(f'  ... and {int(classifications["needs_review"].sum()) - 40:,} more '
        f'(see element_classifications.parquet)')

PATH_REPORT.write_text('\n'.join(report_lines))
print()
print(f'Report written to {PATH_REPORT}')

NOTEBOOK 03b: ELEMENT CLASSIFICATION REPORT
unique elements classified : 40,877
stockiness rows enriched   : 456,159
gold-set accuracy          : 94.5% (121/128)

Taxonomy distribution (unique elements):
  character_type         7,572  [narrative]
  setting                3,375  [narrative]
  object                 5,166  [narrative]
  event                  8,098  [narrative]
  theme                  3,056  [narrative]
  relationship             799  [narrative]
  character_name         4,364  [FILTERED]
  place_name             2,745  [FILTERED]
  proper_noun_other      1,433  [FILTERED]
  generic_or_empty       4,269  [FILTERED]

flagged ambiguous   : 5,844
flagged split-sense : 2,845
flagged needs_review: 545
resolved at Tier 2  : 16,339

PER-GENRE TOP 15 BEFORE vs AFTER FILTER (ranked by ensemble)

--- Action ---
  before: kill, man, escape, fight, shoot, gang, defeat, destroy, save, officer, capture, battle, fight, attack, team
  after : kill, escape, fight, shoot, gang, defeat, 

## Outputs produced

- **`element_classifications.parquet`** - one row per unique `(element, element_type)` pair, with `element_taxonomy` (ten categories), `is_narrative_element`, `classification_confidence`, `is_ambiguous`, `is_split_sense`, `classification_tier`, `rule_prior`, `rule_prior_agrees`, and `needs_review`.
- **`stockiness_long_classified.parquet`** - the Notebook 03 stockiness table with the classification columns joined on. This is the canonical input for every downstream notebook. Row count is identical to `stockiness_long.parquet`; nothing in the original is altered.
- **`classification_report.txt`** - taxonomy distribution, gold-set accuracy, the per-genre top-15 before and after the filter, and the list of elements flagged for review.
- **`cache_tier1/`, `cache_tier2/`, `cache_contexts/`** - per-batch caches. Re-running the notebook reuses them and only processes new or failed batches.

## What feeds where

- **Notebook 03c** (network and combination analysis) builds co-occurrence graphs over nodes where `is_narrative_element` is true, and uses `element_taxonomy` for typed edges.
- **Notebook 04** (plot surprisal) computes per-film mean stockiness over `is_narrative_element` elements only.
- **Notebook 06** (convergent validation) compares the filtered top-K rankings against the expert and human benchmarks.
- **Notebook 07** (court case validation) matches judicially identified elements against the classified, filtered element set.
- **Notebook 08** (year-over-year) runs era-binned analysis on the filtered element set.

## Known limitations

- **Genre-invariance assumption.** One taxonomic label is assigned per `(element, element_type)` pair, treated as genre-invariant. Elements whose category genuinely depends on sense are flagged `is_ambiguous`, and where the senses are clearly split, `is_split_sense`. Downstream analyses that are sensitive to a particular flagged element should handle it explicitly rather than trusting the single label.
- **Polysemy collapse from extraction.** Notebook 02 already merged distinct senses of a token into one element (the flower "rose" and the name "Rose" are one row). Tier 2 example sentences surface this through the `is_split_sense` flag, but the underlying counts cannot be separated without re-running extraction with sense disambiguation, which is out of scope for v1.
- **Gazetteer incompleteness.** The inline name lists are cross-checks, not an exhaustive filter. They are intentionally small and editable. The LLM remains the primary classifier; a gazetteer miss does not mean an element escaped classification, only that one cross-check did not fire.
- **The event / theme boundary is porous.** Words like "betrayal" or "sacrifice" sit between a discrete plot beat and an abstract throughline. Both are `is_narrative_element` true, so the porousness does not affect filtering. It does add noise to the stratified-by-category analysis, which the paper should acknowledge.

## Re-running

The notebook is idempotent. Cached batches are skipped, so a re-run after a disconnect resumes where it stopped. To force a clean re-run, delete the `cache_tier1/`, `cache_tier2/`, and `cache_contexts/` directories. Set `DRY_RUN = True` in the config cell to exercise the full pipeline on two batches per tier first.